In [1]:
import os
os.environ['RUCIO_CONFIG'] = '/home/jovyan/work/ML4Fires/rucio.cfg'

import xarray as xr
import numpy as np
from typing import Any
from rucio.client.client import Client
from types import SimpleNamespace
#from rucio.client.uploadclient import UploadClient
rucio = Client()


In [4]:
variable = "*"
frequency = "*"
model = "CMCC-ESM2"
scenario = "*"
filters = variable + "_*_" + model + "_" + scenario + "_*.nc"
listing = rucio.list_dids(scope="abennasser", filters={"name": filters}, did_type="file")
#filelist = list(listing)
import pprint
#pprint.pprint(len(sorted(filelist)))
rse="VEGA-DCACHE"
dids = []
for l in listing:
    dids.append({"scope": "abennasser", "name": l})
replicas = rucio.list_replicas(
    dids=dids,
    schemes=["file"],
    rse_expression=rse
)
datapath=[]
for r in replicas:
        if rse in r["rses"]:
            lfilepath=r["rses"][rse][0]
            filepath = lfilepath.replace('file://localhost', '')
            datapath.append(filepath)
pprint.pprint((datapath))

2025-06-04 09:23:28,017	ERROR	ConnectionError: HTTPSConnectionPool(host='rucio-intertwin-testbed.desy.de', port=443): Max retries exceeded with url: /dids/abennasser/dids/search?type=file&filters=%5B%7B%27name%27%3A+%27%2A_%2A_CMCC-ESM2_%2A_%2A.nc%27%7D%5D&long=False&recursive=False (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7fcc29933070>: Failed to resolve 'rucio-intertwin-testbed.desy.de' ([Errno -3] Temporary failure in name resolution)"))


['/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20150101-20161231.nc',
 '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20170101-20181231.nc',
 '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20190101-20201231.nc',
 '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20210101-20221231.nc',
 '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20230101-20241231.nc',
 '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20250101-20261231.nc',
 '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20270101-20281231.nc',
 '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20290101-20301231.nc',
 '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20310101-20321231.nc',
 '/dcache/sling.si/

In [35]:
# Initialize Rucio client
_rucio = Client()

def make_8day_windows(year_range: tuple[int, int]) -> np.ndarray:
    """
    Generate non-overlapping 8-day windows (inclusive) over the given year range.
    """
    start_year, end_year = year_range
    start_date = np.datetime64(f"{start_year}-01-01")
    end_date = np.datetime64(f"{end_year}-12-31")

    windows = []
    current = start_date
    one_day = np.timedelta64(1, 'D')
    eight_days = np.timedelta64(8, 'D')
    while current <= end_date:
        window_end = current + eight_days - one_day
        if window_end > end_date:
            window_end = end_date
        windows.append((current, window_end))
        current = window_end + one_day

    return np.array(windows, dtype='datetime64[D]')

def get_cmip6_files(
    scope: str,
    rse: str,
    model_name: str,
    scenario: str,
    year_range: tuple[int,int],
    drivers: dict[str, Any],
) -> tuple[dict[str, list[str]], np.ndarray]:
    """
    Discover CMIP6 local file paths in Rucio for each variable in `drivers`.

    Returns
    -------
    cmip6_var_files : dict var_name → list of local file paths
    windows         : np.ndarray of 8-day date windows
    """
    windows = make_8day_windows(year_range)
    cmip6_var_files: dict[str, list[str]] = {}

    for var, cfg in drivers.items():
        cmip6_var_files[var] = []

        if cfg.type == "dynamic":
            pattern = f"{var}_*_{model_name}_{scenario}_*.nc"
            dids_list = list(_rucio.list_dids(
                scope=scope,
                filters={"name": pattern},
                did_type="file"
            ))
            if not dids_list:
                raise FileNotFoundError(f"No dynamic files found for {var} with pattern {pattern}")

            # resolve to physical paths
            replicas = _rucio.list_replicas(
                dids=[{"scope": scope, "name": d} for d in dids_list],
                schemes=["file"],
                rse_expression=rse
            )
            paths = [
                rep["rses"][rse][0].replace("file://localhost", "")
                for rep in replicas if rse in rep["rses"]
            ]

            for p in sorted(paths):
                fn = os.path.basename(p)
                datestr = fn.rsplit("_", 1)[-1].removesuffix(".nc")
                start_s, end_s = datestr.split("-")
                start = np.datetime64(f"{start_s[:4]}-{start_s[4:6]}-{start_s[6:]}")
                end   = np.datetime64(f"{end_s[:4]}-{end_s[4:6]}-{end_s[6:]}")
                if any((w[0] >= start and w[1] <= end) for w in windows):
                    cmip6_var_files[var].append(p)

        else:
            # static variable: pick exactly one file
            pattern = f"{var}_*_{model_name}_{scenario}_*.nc"
            dids_list = list(_rucio.list_dids(
                scope=scope,
                filters={"name": pattern},
                did_type="file"
            ))
            if not dids_list:
                raise FileNotFoundError(f"No static file found for {var} with pattern {pattern}")

            # resolve that one DID
            reps = _rucio.list_replicas(
                dids=[{"scope": scope, "name": dids_list[0]}],
                schemes=["file"],
                rse_expression=rse
            )
            # pick the first replica on the requested RSE
            rep = next(r for r in reps if rse in r["rses"])
            p = rep["rses"][rse][0].replace("file://localhost", "")
            cmip6_var_files[var] = [p]

    print("Loading the following CMIP6 data files from CERN")
    for v, files in cmip6_var_files.items():
        print(f"{v}: {files}")

    return cmip6_var_files, windows

# ——— Example usage ———

from types import SimpleNamespace

drivers_cfg = {
    "lai":     SimpleNamespace(type="dynamic"),
    "tasmax":  SimpleNamespace(type="dynamic"),
    "hur":   SimpleNamespace(type="dynamic"),
}



In [36]:
# to match only ssp126:
files126, windows = get_cmip6_files(
    scope="abennasser",
    rse="VEGA-DCACHE",
    model_name="CMCC-ESM2",
    scenario="ssp126",        # only ssp126
    year_range=(2015, 2064),
    drivers=drivers_cfg
)

Loading the following CMIP6 data files…
lai: ['/dcache/sling.si/projects/intertwin/abennasser/lai_Eday_CMCC-ESM2_ssp126_r1i1p1f1_gn_20150101-20641231.nc']
tasmax: ['/dcache/sling.si/projects/intertwin/abennasser/tasmax_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20150101-20391231.nc', '/dcache/sling.si/projects/intertwin/abennasser/tasmax_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20400101-20641231.nc']
hur: ['/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20150101-20161231.nc', '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20170101-20181231.nc', '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20190101-20201231.nc', '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20210101-20221231.nc', '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20230101-20241231.nc', '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ss

### without put any scenario

In [5]:
def make_8day_windows(year_range: tuple[int,int]) -> np.ndarray:
    """
    Build an array of [start, end] np.datetime64 windows each 8 days long,
    from Jan 1 of year_range[0] up to Dec 31 of year_range[1].
    """
    start = np.datetime64(f"{year_range[0]}-01-01")
    end   = np.datetime64(f"{year_range[1]}-12-31")
    windows = []
    current = start
    one_day = np.timedelta64(1, "D")
    eight_days = np.timedelta64(8, "D")
    
    while current <= end:
        # define an 8-day window (current through current+7 days)
        window_end = current + (eight_days - one_day)
        # clip to final calendar date
        if window_end > end:
            window_end = end
        windows.append((current, window_end))
        # advance by 8 days
        current = current + eight_days

    return np.array(windows, dtype="datetime64[ns]")

# quick check that we indeed step every 8 days:
#for w in make_8day_windows((2020, 2020))[:5]:
#    print(w)


In [6]:
def get_cmip6_files(
    scope: str,
    rse: str,
    model_name: str,
    scenario: str,
    year_range: tuple[int,int],
    drivers: dict[str, Any],
) -> tuple[dict[str, list[str]], np.ndarray]:
    """
    Discover CMIP6 files for each driver variable via Rucio and filter them
    to cover the requested 8-day windows.

    Parameters
    ----------
    scope       : Rucio scope (e.g. "abennasser")
    rse         : preferred Rucio storage endpoint (e.g. "VEGA-DCACHE")
    model_name  : CMIP6 model ID (e.g. "CMCC-ESM2")
    scenario    : SSP scenario (e.g. "ssp126") or "" to accept any scenario
    year_range  : (start_year, end_year) tuple, inclusive
    drivers     : dict mapping variable name → config object with attribute `.type`
                  where `.type` is "dynamic" or anything else for static files

    Returns
    -------
    cmip6_var_files : dict mapping var_name → list of local file paths
    windows         : ndarray of shape (n_windows, 2) of the 8-day windows
    """
    rucio = Client()
    windows = make_8day_windows(year_range)
    cmip6_var_files: dict[str, list[str]] = {}

    for var, cfg in drivers.items():
        cmip6_var_files[var] = []
        if cfg.type == "dynamic":
            # build Rucio list_dids pattern
            if scenario:
                pattern = f"{var}_*_{model_name}_{scenario}_*.nc"
            else:
                pattern = f"{var}_*_{model_name}_*.nc"

            # list matching DIDs
            dids = list(rucio.list_dids(scope=scope, filters={"name": pattern}, did_type="file"))

            # resolve replicas to file:// paths
            reps = rucio.list_replicas(
                dids=[{"scope": scope, "name": d} for d in dids],
                schemes=["file"],
                rse_expression=rse
            )

            # extract local paths
            paths = [
                rep["rses"][rse][0].replace("file://localhost", "")
                for rep in reps if rse in rep["rses"]
            ]

            # filter by 8-day windows
            for p in sorted(paths):
                fn = os.path.basename(p)
                datestr = fn.rsplit("_", 1)[-1].removesuffix(".nc")
                start_s, end_s = datestr.split("-")
                start = np.datetime64(f"{start_s[:4]}-{start_s[4:6]}-{start_s[6:]}")
                end   = np.datetime64(f"{end_s[:4]}-{end_s[4:6]}-{end_s[6:]}")
                # keep file if it fully covers any window
                if any((w[0] >= start and w[1] <= end) for w in windows):
                    cmip6_var_files[var].append(p)

        else:
            # static → pick the first available file
            if scenario:
                pattern = f"{var}_*_{model_name}_{scenario}_*.nc"
            else:
                pattern = f"{var}_*_{model_name}_*.nc"

            dids = list(rucio.list_dids(scope=scope, filters={"name": pattern}, did_type="file"))
            if not dids:
                raise FileNotFoundError(f"No static file found for variable '{var}' with pattern {pattern}")

            # resolve a single DID
            reps = rucio.list_replicas(
                dids=[{"scope": scope, "name": dids[0]}],
                schemes=["file"],
                rse_expression=rse
            )
            rep = next(r for r in reps if rse in r["rses"])
            p = rep["rses"][rse][0].replace("file://localhost", "")
            cmip6_var_files[var] = [p]

    print("Loading the following CMIP6 data files from CERN")
    for v, files in cmip6_var_files.items():
        print(f"{v}: {files}")

    return cmip6_var_files, windows




drivers_cfg = {
    "lai":     SimpleNamespace(type="dynamic"),
    "tasmax":  SimpleNamespace(type="dynamic"),
    "hur":   SimpleNamespace(type="dynamic"),
}



In [8]:

#from ourconfig import drivers

files_dict, date_windows = get_cmip6_files(
    scope="abennasser",
    rse="VEGA-DCACHE",
    model_name="CMCC-ESM2",
    scenario="",          # or "ssp126" to accept any scenario
    year_range=(2015, 2020),
    drivers=drivers_cfg
)

Loading the following CMIP6 data files from CERN
lai: ['/dcache/sling.si/projects/intertwin/abennasser/lai_Eday_CMCC-ESM2_ssp126_r1i1p1f1_gn_20150101-20641231.nc', '/dcache/sling.si/projects/intertwin/abennasser/lai_Eday_CMCC-ESM2_ssp245_r1i1p1f1_gn_20150101-20641231.nc', '/dcache/sling.si/projects/intertwin/abennasser/lai_Eday_CMCC-ESM2_ssp585_r1i1p1f1_gn_20150101-20641231.nc']
tasmax: ['/dcache/sling.si/projects/intertwin/abennasser/tasmax_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20150101-20391231.nc', '/dcache/sling.si/projects/intertwin/abennasser/tasmax_day_CMCC-ESM2_ssp245_r1i1p1f1_gn_20150101-20391231.nc', '/dcache/sling.si/projects/intertwin/abennasser/tasmax_day_CMCC-ESM2_ssp585_r1i1p1f1_gn_20150101-20391231.nc']
hur: ['/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20150101-20161231.nc', '/dcache/sling.si/projects/intertwin/abennasser/hur_day_CMCC-ESM2_ssp126_r1i1p1f1_gn_20170101-20181231.nc', '/dcache/sling.si/projects/intertwin/abennasser/hur_da

## run after this cell

In [1]:
import numpy as np
import xarray as xr
import toml
import munch
from tqdm import tqdm
import torch
import datetime
import zarr

import warnings
warnings.filterwarnings("ignore")

import os
os.environ['RUCIO_CONFIG'] = '/home/jovyan/work/ML4Fires/rucio.cfg'

from typing import Any
from rucio.client.client import Client
from types import SimpleNamespace
#from rucio.client.uploadclient import UploadClient
rucio = Client()

from Fires._utilities.utils_mlflow import load_model_from_mlflow
from Fires._utilities.utils_inference import get_cmip6_inference,load_cmip6_files_from_config, get_cmip6_files_rucio,_get_file_list

In [2]:
variable = "*"
frequency = "*"
model = "CMCC-ESM2"
scenario = "*"
filters = variable + "_*_" + model + "_" + scenario + "_*.nc"
listing = rucio.list_dids(scope="abennasser", filters={"name": filters}, did_type="file")
#filelist = list(listing)
import pprint
#pprint.pprint(len(sorted(filelist)))
rse="VEGA-DCACHE"
dids = []
for l in listing:
    dids.append({"scope": "abennasser", "name": l})
replicas = rucio.list_replicas(
    dids=dids,
    schemes=["file"],
    rse_expression=rse
)
datapath=[]
for r in replicas:
        if rse in r["rses"]:
            lfilepath=r["rses"][rse][0]
            filepath = lfilepath.replace('file://localhost', '')
            datapath.append(filepath)


2025-06-05 09:56:13,613	ERROR	ConnectionError: HTTPSConnectionPool(host='rucio-intertwin-testbed.desy.de', port=443): Max retries exceeded with url: /dids/abennasser/dids/search?type=file&filters=%5B%7B%27name%27%3A+%27%2A_%2A_CMCC-ESM2_%2A_%2A.nc%27%7D%5D&long=False&recursive=False (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7f3550f38880>: Failed to resolve 'rucio-intertwin-testbed.desy.de' ([Errno -2] Name or service not known)"))


In [3]:
from IPython.display import display, clear_output
import ipywidgets as widgets

# -----------------------------
# UI widgets
# -----------------------------
scenario = widgets.Dropdown(
    options=[
        ('SSP126', 'ssp126'),
        ('SSP245', 'ssp245'),
        ('SSP370', 'ssp370'),
        ('SSP585', 'ssp585')
    ],
    value='ssp126',
    description='CMIP6 Scenario:',
    style={'description_width': '150px'}
)

year_range = widgets.IntRangeSlider(
    value=[2030, 2035],
    min=2015,
    max=2100,
    step=1,
    description='Year Range:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

run_button = widgets.Button(
    description='Load CMIP6 Data',
    button_style='success',
    icon='cloud-download'
)

output = widgets.Output()

# -----------------------------
# Callback using your final function
# -----------------------------
def on_run_button_clicked(b):
    with output:
        clear_output()
        print(f"📌 Scenario selected: {scenario.value}")
        print(f"📆 Year range selected: {year_range.value}")
        
        try:
            files_dict, date_windows = load_cmip6_files_from_config(
                scenario=scenario.value,
                year_range=tuple(year_range.value)
            )
            print("✅ Files loaded successfully.")
            for var, files in files_dict.items():
                print(f"🔹 {var}: {len(files)} file(s)")
        except Exception as e:
            print(f"❌ Error loading files:\n{e}")

# Hook up the callback
run_button.on_click(on_run_button_clicked)

# -----------------------------
# Display everything
# -----------------------------
display(widgets.VBox([
    widgets.HBox([scenario, year_range]),
    run_button,
    output
]))
